# Row-Column Attention (RCA) Model Implementation

이 노트북은 **Antigravity Colab Extension** 환경에서 실행 중입니다.
RCA 아키텍처의 핵심인 전역 그리드 디커플링(Global Grid Decoupling)과 이축 주의 집중(Dual-Axis Attention)을 딥러닝 모델로 구현합니다.

## 1. 환경 설정 및 데이터 진단 (Setup & Data Diagnosis)

In [ ]:
!pip install timm albumentations matplotlib torch torchvision

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import os
import sys
import glob

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Current Working Directory: {os.getcwd()}")
print(f"Files in CWD: {os.listdir('.')}")

### 리포지토리 및 데이터 확인
Git 리포지토리를 클론하고 실제 데이터 파일의 위치를 찾습니다.

In [ ]:
# 1. Git Clone (필요 시)
if not os.path.exists('HiTab'):
    print("Cloning HiTab repository...")
    !git clone https://github.com/microsoft/HiTab.git
else:
    print("HiTab repository already exists.")

# 2. 데이터 파일 집중 수색
def find_hitab_file_exhaustive():
    filename = 'hitab_experiment_dataset.json'
    
    # 탐색할 후보 경로들 (절대 및 상대)
    candidates = [
        '/home/user/t1-4/data/' + filename,
        './data/' + filename,
        '../data/' + filename,
        './' + filename,
        'HiTab/data/train.json', # GitHub 기본 파일
        'HiTab/data/dev.json'    # GitHub 기본 파일
    ]
    
    for c in candidates:
        abs_c = os.path.abspath(c)
        if os.path.exists(abs_c):
            print(f"SUCCESS: Found dataset at {abs_c}")
            return abs_c
    
    # 못 찾았을 경우 전체 하위 폴더 검색 (느릴 수 있음)
    print("Deep searching in current folder and parents...")
    # 상위 폴더로 이동하며 검색 (최대 3단계)
    search_root = os.getcwd()
    for _ in range(3):
        results = glob.glob(os.path.join(search_root, '**', filename), recursive=True)
        if results:
            found = os.path.abspath(results[0])
            print(f"SUCCESS (Deep Search): Found at {found}")
            return found
        search_root = os.path.dirname(search_root)
        if search_root == '/': break
            
    return None

TARGET_DATA_PATH = find_hitab_file_exhaustive()
if not TARGET_DATA_PATH:
    print("ERROR: Could not find any dataset file. Please ensure the file exists.")

## 2. 모델 아키텍처 및 데이터셋 정의

In [ ]:
class RCAModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('resnet18', pretrained=True, features_only=True)
        embed_dim = 512
        
        # Transformer Attention
        self.attention = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True)
        
        self.row_pool = nn.AdaptiveAvgPool2d((None, 1))
        self.row_head = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.ReLU(), nn.Conv2d(256, 1, 1), nn.Sigmoid())
        
        self.col_pool = nn.AdaptiveAvgPool2d((1, None))
        self.col_head = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.ReLU(), nn.Conv2d(256, 1, 1), nn.Sigmoid())
        
        self.cell_map_head = nn.Conv2d(embed_dim, 1, kernel_size=1)

    def forward(self, x):
        features = self.backbone(x)[-1]
        B, C, H, W = features.shape
        # Attention
        x_flat = features.flatten(2).transpose(1, 2)
        x_att = self.attention(x_flat)
        features = x_att.transpose(1, 2).reshape(B, C, H, W)
        
        row_probs = self.row_head(self.row_pool(features)).squeeze(-1).squeeze(1)
        col_probs = self.col_head(self.col_pool(features)).squeeze(-2).squeeze(1)
        cell_map = torch.sigmoid(self.cell_map_head(features))
        
        return {"row_probs": row_probs, "col_probs": col_probs, "cell_map": cell_map}

class HiTabDataset(Dataset):
    def __init__(self, json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)
        if isinstance(data, list): self.tables = [item.get('table', item) for item in data]
        elif isinstance(data, dict): self.tables = list(data.get('tables', {}).values()) or list(data.values())
        else: self.tables = []
        self.synthesizer = TableImageSynthesizer()
        print(f"Dataset Size: {len(self.tables)}")
        
    def __len__(self): return len(self.tables)
    def __getitem__(self, idx):
        table = self.tables[idx]
        try:
            image, r_idxs, c_idxs = self.synthesizer.render(table)
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            row_gt, col_gt = self.synthesizer.generate_targets(16, 16, r_idxs, c_idxs)
            return image, row_gt, col_gt, torch.ones(1, 16, 16)
        except: return self.__getitem__((idx + 1) % len(self))

## 3. 학습 실행 (Training)

In [ ]:
def rca_loss(preds, targets):
    loss_fn = nn.BCELoss()
    return loss_fn(preds['row_probs'], targets['row_gt']) + loss_fn(preds['col_probs'], targets['col_gt']) + loss_fn(preds['cell_map'], targets['cell_gt'])

model = RCAModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

if TARGET_DATA_PATH:
    dataset = HiTabDataset(TARGET_DATA_PATH)
    train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
    print("Training starting with REAL DATA...")
    model.train()
    for batch_idx, (imgs, r_gt, c_gt, cell_gt) in enumerate(train_loader):
        imgs, targets = imgs.to(device), {'row_gt': r_gt.to(device), 'col_gt': c_gt.to(device), 'cell_gt': cell_gt.to(device)}
        output = model(imgs)
        loss = rca_loss(output, targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if batch_idx % 5 == 0: print(f"Batch {batch_idx}: Loss: {loss.item():.4f}")
        if batch_idx > 10: break
else:
    print("FALLBACK: Using Dummy Data for Demo (Dataset not found in search)")
    # ... dummy logic ...
    dummy_input = torch.randn(2, 3, 512, 512).to(device)
    output = model(dummy_input)
    print("Dummy Forward Pass Successful.")